# BENTO Edge AI — Training บน Google Colab (ฟรี ไม่ต้องมี Docker)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tesaiot/tesa-qualification-program/blob/main/courses/edge-ai-developer/shared/training/notebooks/train_edge_ai.ipynb)

โน้ตบุ๊กนี้ทำ **เสาที่ 4 (Training)** ของคอร์ส Edge AI Developer ให้ครบในที่เดียว — 
จาก *ข้อมูล* → *โมเดล Keras* → **`model_int8.tflite`** ไฟล์เดียวที่ deploy ได้ทั้ง 
**MCU (Ethos-U55) · เบราว์เซอร์ · PC · Cortex-A** = *train once, run everywhere*.

รันทีละเซลล์ (Shift+Enter). Colab มี TensorFlow ให้แล้ว ไม่ต้องลง Docker.

> เนื้อหาตรงกับ `shared/training/train.py` + `dataset_tools.py` ในคอร์ส — 
> ต่างแค่รันบนคลาวด์ และ synthesize ข้อมูลให้เอง (ยังไม่ต้องมีบอร์ด).


## 0) ตรวจสภาพแวดล้อม


In [ ]:
import numpy as np, tensorflow as tf
print('TensorFlow', tf.__version__)
print('NumPy', np.__version__)

## 1) Dataset — สังเคราะห์สัญญาณ IMU 3 ท่า
`idle` = อยู่นิ่ง · `circle` = หมุนเป็นวง · `shaking` = สั่นแรง (variance สูง).
โครงเดียวกับ `dataset_tools.synthesize()` — 6 แกน (ax,ay,az,gx,gy,gz) ที่ ~50 Hz. ต่างกันที่หน่วย: เซลล์นี้ใช้ accel หน่วย g (วางนิ่ง az = 1) ส่วนบอร์ดและ `dataset_tools.py` ใช้ m/s² (วางนิ่ง az ≈ 9.81) ฝึกกับข้อมูลสังเคราะห์ได้เพราะ normalize ก่อน แต่โมเดลที่จะใช้กับบอร์ดต้องฝึกจาก CSV ของบอร์ด


In [ ]:
CLASSES  = ['idle', 'circle', 'shaking']   # ตรงกับโมเดล Motion บนบอร์ด
CHANNELS = ['ax','ay','az','gx','gy','gz']
WIN, HOP, FS = 50, 25, 50                  # หน้าต่าง 1 วินาที, ซ้อน 50%

def synthesize(per_class=1200, seed=0):
    rng = np.random.default_rng(seed)
    xs, ys = [], []
    for cls in range(len(CLASSES)):
        for i in range(per_class):
            t = i / FS
            if cls == 0:      # idle: แรงโน้มถ่วง + noise เล็ก
                s = [0,0,1,0,0,0] + rng.normal(0, 0.02, 6)
            elif cls == 1:    # circle: accel หมุนเป็นวง
                s = [0.6*np.sin(2*np.pi*1.5*t), 0.6*np.cos(2*np.pi*1.5*t), 1,
                     40*np.sin(2*np.pi*1.5*t), 40*np.cos(2*np.pi*1.5*t), 5] + rng.normal(0,0.05,6)
            else:             # shaking: variance สูงทุกแกน
                s = [0,0,1,0,0,0] + rng.normal(0, 0.8, 6)
            xs.append(s); ys.append(cls)
    return np.asarray(xs, np.float32), np.asarray(ys, np.int64)

samples, labels = synthesize()
print('samples', samples.shape, 'labels', labels.shape)

## 2) Windowing + Normalize + Split
ตัดสตรีมเป็นหน้าต่างซ้อนกัน, standardize ต่อแกน (fit เฉพาะ train), แบ่ง train/val/test แบบ stratified.

**สำคัญ:** เก็บ `mean/std` ไว้ เพราะฝั่งบอร์ด/เบราว์เซอร์ต้อง normalize ด้วยค่าเดียวกัน (จุดพังเงียบที่พบบ่อย).


In [ ]:
def make_windows(x, y, win=WIN, hop=HOP):
    X, Y = [], []
    for s in range(0, len(x)-win+1, hop):
        X.append(x[s:s+win])
        Y.append(np.bincount(y[s:s+win], minlength=len(CLASSES)).argmax())
    return np.asarray(X, np.float32), np.asarray(Y, np.int64)

def split(X, y, val=.15, test=.15, seed=0):
    rng = np.random.default_rng(seed); tr,va,te=[],[],[]
    for c in np.unique(y):
        ii = rng.permutation(np.where(y==c)[0]); n=len(ii)
        nte,nva = int(n*test), int(n*val)
        te+=list(ii[:nte]); va+=list(ii[nte:nte+nva]); tr+=list(ii[nte+nva:])
    rng.shuffle(tr); rng.shuffle(va); rng.shuffle(te)
    p=lambda s:(X[s],y[s]); return p(np.array(tr)),p(np.array(va)),p(np.array(te))

X, y = make_windows(samples, labels)
(Xtr,ytr),(Xva,yva),(Xte,yte) = split(X, y)
mean = Xtr.reshape(-1,6).mean(0); std = Xtr.reshape(-1,6).std(0)+1e-6
norm = lambda A:(A-mean)/std
Xtr,Xva,Xte = norm(Xtr),norm(Xva),norm(Xte)
print('train/val/test windows:', len(Xtr), len(Xva), len(Xte))

## 3) โมเดล — Conv1D เล็ก ๆ (พอดีกับ NPU ของ MCU)
Conv1D ไล่ตามเวลา → global pooling → dense head. ทุก op เป็นมิตรกับ Ethos-U55.

**คณิตหลังฉาก:** ชั้นสุดท้าย softmax $\hat{y}_i = e^{z_i}/\sum_j e^{z_j}$, 
loss = cross-entropy $L=-\sum_i y_i\log\hat{y}_i$, เรียนด้วย gradient descent $\theta \leftarrow \theta-\eta\nabla_\theta L$.


In [ ]:
def build_model(win, chans, n):
    return tf.keras.Sequential([
        tf.keras.layers.Input((win, chans)),
        tf.keras.layers.Conv1D(16, 5, padding='same', activation='relu'),
        tf.keras.layers.MaxPooling1D(2),
        tf.keras.layers.Conv1D(32, 3, padding='same', activation='relu'),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(n, activation='softmax'),
    ])

model = build_model(WIN, len(CHANNELS), len(CLASSES))
model.compile('adam', 'sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 4) ฝึก (fit) — ดู loss ลด / accuracy เพิ่มทุก epoch


In [ ]:
hist = model.fit(Xtr, ytr, validation_data=(Xva, yva),
                 epochs=25, batch_size=32, verbose=2)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(hist.history['accuracy'], label='train')
plt.plot(hist.history['val_accuracy'], label='val')
plt.xlabel('epoch'); plt.ylabel('accuracy'); plt.legend(); plt.title('learning curve'); plt.show()
_, acc = model.evaluate(Xte, yte, verbose=0)
print('float32 test accuracy: %.3f' % acc)

## 5) Quantize → int8 (full-integer)
NPU เร่งเฉพาะ op ที่ quantize แล้ว, และ int8 ก็รันได้ทั้งเบราว์เซอร์/Cortex-A → int8 = ตัวหารร่วมของทุกเป้าหมาย.
ป้อน *representative dataset* ให้ converter คาลิเบรตช่วงค่า activation.


In [ ]:
def to_int8_tflite(model, X_repr, path='model_int8.tflite'):
    def representative():
        for i in range(min(200, len(X_repr))):
            yield [X_repr[i:i+1].astype(np.float32)]
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = representative
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = tf.int8
    conv.inference_output_type = tf.int8
    blob = conv.convert()
    open(path,'wb').write(blob)
    print('wrote', path, '(%d bytes)' % len(blob)); return path

path = to_int8_tflite(model, Xtr)
np.savez(path + '.norm.npz', mean=mean, std=std)   # ส่งไปฝั่ง board/browser ด้วย

## 6) ตรวจไฟล์ int8 บน test set — accuracy ต้องใกล้ float32
รันผ่าน TFLite interpreter (dequantize input/output ตาม scale/zero_point) แล้วเทียบกับ ground truth.


In [ ]:
it = tf.lite.Interpreter(model_path=path); it.allocate_tensors()
inp, out = it.get_input_details()[0], it.get_output_details()[0]
si, zi = inp['quantization']; so, zo = out['quantization']
ok = 0
for k in range(len(Xte)):
    q = np.round(Xte[k]/si + zi).astype(np.int8)[None]
    it.set_tensor(inp['index'], q); it.invoke()
    o = it.get_tensor(out['index'])[0].astype(np.float32)
    if (o.argmax()) == yte[k]: ok += 1
print('int8 test accuracy: %.3f' % (ok/len(Xte)))

## 7) ดาวน์โหลด → เอาไปรัน 'ทุกที่'
- **MCU:** `quantize_vela.sh` → Vela → Ethos-U55 (ดูบทเรียน 5.8–5.9)
- **เบราว์เซอร์:** `convert_web.py` → ONNX-Runtime-Web (ดูบทเรียน 5.6–5.7 แล้วตรวจ parity กับ `eval_pc.py` ด้วยชุดทดสอบเดียวกัน)
- **Cortex-A:** `ai-edge-litert` บน RPi/Jetson (รัน `eval_pc.py` ตัวเดียวกับบน PC)

ไฟล์เดียว `model_int8.tflite` — เป้าหมายเดียวกันทั้งหมด.


In [ ]:
try:
    from google.colab import files
    files.download(path)
    files.download(path + '.norm.npz')
except Exception as e:
    print('ไม่ได้อยู่บน Colab — ไฟล์อยู่ที่', path)